# Day 2: Building a Real-World Telecom RAG System

In [25]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

# Multilingual embeddings (Crucial for matching Arabic queries to English text)
print("Loading local embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Load the Knowledge Base
print("Loading Knowledge Base...")
loader = TextLoader('data/Telecom_Internal_KB.txt', encoding='utf-8')
documents = loader.load()
print(f"✅ Successfully loaded {len(documents)} document(s).")

Loading local embedding model...


'[WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
'[WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions' thrown while requesting HEAD https://huggingface.co/sentence-transfor

Loading Knowledge Base...
✅ Successfully loaded 1 document(s).


In [38]:
import re

def parse_error_codes(document_text: str) -> dict:
    """Reads the raw document text and builds a dict mapping each error code to its description and resolution."""
    pattern = re.compile(
        r'#### Error Code (E-\d+)\s*\n\*\*Description:\*\*\s*(.+?)\s*\n\*\*Resolution Protocol:\*\*\s*(.+?)(?=\n####|\n##|\Z)',
        re.DOTALL
    )
    db = {}
    for match in pattern.finditer(document_text):
        code, desc, resolution = match.groups()
        db[code] = {"Description": desc.strip(), "Resolution Protocol": resolution.strip()}
    return db

error_codes_db = parse_error_codes(documents[0].page_content)
print(f"Extracted {len(error_codes_db)} error codes")

def get_exact_code_context(question: str) -> str:
    """Finds any E-XXX code literally mentioned in the question and returns its guaranteed-correct info."""
    codes_found = re.findall(r'E-\d{3}', question)
    if not codes_found:
        return ""
    lines = []
    for code in codes_found:
        info = error_codes_db.get(code)
        if info:
            lines.append(
                f"[Code {code} - confirmed from database]\n"
                f"Description: {info['Description']}\n"
                f"Resolution: {info['Resolution Protocol']}"
            )
    return "\n\n".join(lines)

Extracted 300 error codes


In [56]:
def extract_section(document_text: str, header: str) -> str:
    """Extracts one full ## section by its exact heading text, stopping at the next ## or end of doc."""
    pattern = re.compile(
        rf'(## {re.escape(header)}.*?)(?=\n## |\Z)',
        re.DOTALL
    )
    match = pattern.search(document_text)
    return match.group(1).strip() if match else ""

# This policy section is short (~700 chars) and always relevant to outage complaints —
# so we inject it in full every time instead of relying on chunked semantic search for it.
sla_policy_text = extract_section(
    documents[0].page_content,
    "1. General Service Level Agreement (SLA) & Dispatch Policies"
)
print(f"SLA policy section extracted: {len(sla_policy_text)} characters")

SLA policy section extracted: 665 characters


## Structure-Aware Markdown Chunking
Improve the original chunking strategy by using the knowledge base's existing Markdown hierarchy (`#`, `##`, `###`) to split content along meaningful section boundaries instead of relying only on character length. This preserves the context and metadata of related sections, such as keeping a router model's specifications logically grouped, while recursive splitting is used afterward only for sections that are still too large.

In [39]:
# Split the text into chunks
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# Step 1: Split by headers (# ## ###) first
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"), 
]
markdown_splitter = MarkdownHeaderTextSplitter(        
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

# Apply it to the document text (not directly to the documents, it needs the text as a string)
md_chunks = markdown_splitter.split_text(documents[0].page_content)

# Step 2: Any section that is still too large (like a long SLA section) is split into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)
chunks = text_splitter.split_documents(md_chunks)

print(f"✅ Split into {len(chunks)} chunks.")
print(f"🔍 Sample chunk metadata: {chunks[6].metadata}")
print(f"🔍 Sample chunk content: \n{chunks[5].page_content}")

✅ Split into 703 chunks.
🔍 Sample chunk metadata: {'Header 1': 'Telecom Egypt Internal Technical Support Knowledge Base (Confidential)', 'Header 2': '2. Hardware Specifications & Router Guides', 'Header 3': 'Router Model: VDF-ZTE-2025X3'}
🔍 Sample chunk content: 
- **Troubleshooting Step 1:** Restart router and wait 2 minutes.
- **Troubleshooting Step 2:** Factory reset by holding the reset pin for 10 seconds. Reconfigure with VLAN ID 20.


In [40]:
from langchain_community.vectorstores import FAISS
from tqdm import tqdm

print(f"Starting ingestion of {len(chunks)} chunks into FAISS...")

# We ingest in batches to lower resource costs
batch_size = 50
vectorstore = None

for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding & Indexing Chunks"):
    batch = chunks[i:i + batch_size]
    
    if vectorstore is None:
        # First batch initializes the FAISS index
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        # Subsequent batches are added to the existing index
        vectorstore.add_documents(batch)
        

# Save the FAISS index locally so we don't have to pay/wait to re-embed later
vectorstore.save_local("faiss_telecom_index")
print("\n✅ Ingestion Complete. FAISS index saved locally.")

# Test if the multilingual retrieval actually works!
print("\nTesting semantic search (Arabic Query -> English Document)...")
test_query = "العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام"
print(f"\n{test_query}")

results = vectorstore.similarity_search_with_score(test_query, k=4)
print("\n✅ Top matches retrieved by FAISS:")
print("--------------------------------------------------")

for rank, (doc, score) in enumerate(results, start=1):
    source_section = doc.metadata.get("Header 3", doc.metadata.get("Header 2", "N/A"))
    print(f"#{rank} | Match Score: {score:.4f} | Source Section: {source_section}")
    print(doc.page_content)
    print("--------------------------------------------------")

Starting ingestion of 703 chunks into FAISS...


Embedding & Indexing Chunks: 100%|██████████| 15/15 [00:07<00:00,  1.90it/s]


✅ Ingestion Complete. FAISS index saved locally.

Testing semantic search (Arabic Query -> English Document)...

العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام

✅ Top matches retrieved by FAISS:
--------------------------------------------------
#1 | Match Score: 12.5353 | Source Section: 1. General Service Level Agreement (SLA) & Dispatch Policies
- A Field Technician must be dispatched if the line noise margin is below 6dB or if the DSL light is completely off/blinking for 3 consecutive days.
- The customer must be informed that the technician will contact them within 48 working hours.
- Compensation of 5GB mobile data is authorized ONLY if the outage exceeds 72 hours.
--------------------------------------------------
#2 | Match Score: 12.6735 | Source Section: 4. Cross-Department Escalation Matrix
## 4. Cross-Department Escalation Matrix
- **Billing Issues:** Transfer to 111.
- **Fiber Optic Cuts:** Escalate immediately to Tier 3 Fiber Ops. SLA is 12 hours.
- **Mass Outag

## Metadata-Aware Context Formatting & Optimized Retrieval

Make the RAG pipeline actually use the metadata created during Markdown-based chunking by adding the section name to every retrieved chunk before sending the context to Gemini. This gives the LLM a clear indication of where each piece of information comes from and helps prevent confusion between similar router models. Additionally, replace the original `top-k` retrieval with MMR and reduce the number of final results from 20 to 6, since the new chunks are more complete and information-dense. MMR selects diverse and relevant results from a larger candidate pool (`fetch_k=20`), reducing redundant context, unnecessary noise, and token usage while keeping the retrieved context focused on the customer's issue.

## Prompt Engineering & Response Quality Constraints
Strengthen the system prompt with explicit style and safety constraints to make Gemini behave more like a real Egyptian customer service agent. The prompt prevents company-name leakage, unsupported information, fabricated answers, and overly formal language, while keeping responses concise (around 4–5 sentences) and avoiding repetitive openings for a more natural and consistent customer experience.

In [57]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv

print("Building the Prompt and Gemini RAG Chain...")

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Error: GOOGLE_API_KEY is not found in the .env file — make sure it is added correctly.")
os.environ["GOOGLE_API_KEY"] = api_key

gemini_llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

template = """
أنت موظف خدمة عملاء في مزود خدمة إنترنت (ISP).
مهمتك هي الرد على شكوى العميل بالعامية المصرية بطريقة مهذبة واحترافية.
ممنوع تمامًا:
- ذكر أي اسم شركة اتصالات حقيقي
- ذكر أنك ذكاء اصطناعي أو بوت
- إعطاء رقم/إيميل/رابط غير موجود حرفيًا في السياق
- اختراع معلومة غير موجودة في السياق؛ حوّلها لفريق مختص لو ناقصة
- استخدام ألفاظ رسمية زي "سيادتكم" أو "حضرتكم الموقر"
- تكرار نفس جملة الافتتاح في كل رد
اذكر الحل في نقط مرقمة لو فيه أكتر من خطوة، وابدأ دايمًا بذكر كود الخطأ لو العميل ذكره.
السياق الداخلي:
{context}
شكوى العميل:
{question}
الرد:
"""

prompt = PromptTemplate.from_template(template)

# Improvement 1: Use MMR instead of regular top-k, with a smaller k since the chunks are now more information-dense
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 20}
)

def build_context(question: str) -> str:
    """Builds the final context string sent to Gemini: exact code lookup first, then the semantic search results."""
    docs = retriever.invoke(question)
    parts = []

    parts.append(f"[Section: General SLA Policy]\n{sla_policy_text}")
    
    exact = get_exact_code_context(question)
    if exact:
        parts.append(exact)

    for doc in docs:
        section = doc.metadata.get("Header 4") or doc.metadata.get("Header 3") or doc.metadata.get("Header 2", "General")
        parts.append(f"[Section: {section}]\n{doc.page_content}")

    return "\n\n".join(parts)

rag_chain = (
    {"context": RunnableLambda(build_context), "question": RunnablePassthrough()} | prompt | gemini_llm | StrOutputParser()
)
print("✅ Gemini RAG Chain is ready!")

Building the Prompt and Gemini RAG Chain...
✅ Gemini RAG Chain is ready!


In [45]:
print("Processing the ticket through Gemini...\n")

customer_ticket = """
أنا دافع الفاتورة من يومين أونلاين والفلوس اتخصمت من الفيزا، 
لكن النت لسه مرجعش لحد دلوقتي ومكتوبلي إن الخدمة موقوفة!
"""

print("Agent AI Response (Gemini):")
print("--------------------------------------------------")

# This sends the ticket to the retriever, formats the prompt, and gets the answer from Gemini
response = rag_chain.invoke(customer_ticket)

print(response)
print("--------------------------------------------------")

Processing the ticket through Gemini...

Agent AI Response (Gemini):
--------------------------------------------------


c:\Users\Eslam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


أهلاً بك يا فندم، بعتذر لحضرتك جداً عن الإزعاج وأن الخدمة لسه موقوفة رغم دفع الفاتورة.

بما إن مشكلة حضرتك متعلقة بالفواتير والسداد، هتحتاج تتبع الخطوة دي:

1. التواصل مع قسم الفواتير على رقم 111 لمراجعة عملية الدفع وتأكيد السداد وتفعيل الخدمة لحضرتك.
--------------------------------------------------


In [44]:
print("Processing the ticket through Gemini...\n")

customer_ticket = """
النت شغال بس بطيء جداً وبيظهرلي رسالة على الشاشة فيها كود الخطأ E-204.
أعمل إيه عشان أحل المشكلة دي؟
"""

print("Agent AI Response (Gemini):")
print("--------------------------------------------------")

# This sends the ticket to the retriever, formats the prompt, and gets the answer from Gemini
response = rag_chain.invoke(customer_ticket)

print(response)
print("--------------------------------------------------")

Processing the ticket through Gemini...

Agent AI Response (Gemini):
--------------------------------------------------


c:\Users\Eslam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


كود الخطأ E-204 بيوضح إن فيه نسبة تشويش عالية على الخط، وعشان تحل المشكلة دي وتظبط السرعة معاك، ياريت تطبق الخطوة التالية:

1. تغيير الـ DNS في إعدادات الجهاز أو الراوتر إلى 8.8.8.8.

لو طبقت الخطوة دي ولسه المشكلة مستمرة والنت بطيء، بلغني فوراً عشان أحول المشكلة للفريق المختص يتابع مع حضرتك. تحت أمرك في أي وقت!
--------------------------------------------------


In [58]:
print("Processing the ticket through Gemini...\n")

customer_ticket = """
النت عندي بيقطع من 4 أيام، ولمبة الـ DSL بتنور وتطفي طول الوقت من ساعتها.
كمان ظهرلي كود الخطأ E-354 على الراوتر.
أنا سمعت من صحابي إن الشركة بتدي تعويض باقة نت مجانية في الحالات دي، ده صحيح؟
"""

print("Agent AI Response (Gemini):")
print("--------------------------------------------------")

response = rag_chain.invoke(customer_ticket)

print(response)
print("--------------------------------------------------")

Processing the ticket through Gemini...

Agent AI Response (Gemini):
--------------------------------------------------


c:\Users\Eslam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


بالنسبة لكود الخطأ **E-354** اللي ظهر على الراوتر، ده معناه إن فيه تشويش عالٍ على الخط (Line Noise Too High)، وده السبب في إن لمبة الـ DSL بتنور وتطفي والخدمة متقطعة بقالها 4 أيام.

علشان نحل المشكلة ونطمنك، دي الخطوات والإجراءات المتبعة:

1. يرجى التأكد أولاً من إعادة تشغيل الراوتر (Restart) وفحص جميع الأسلاك والتوصيلات الداخلية في المكان.
2. بما إن المشكلة مستمرة لأكثر من 24 ساعة ولمبة الـ DSL بتنور وتطفي لمدة 4 أيام متواصلة، فإحنا قمنا بتصعيد الشكوى لفريق السنترال المختص وتم طلب زيارة فني صيانة لموقعك.
3. الفني المختص هيتواصل معاك خلال 48 ساعة عمل لمتابعة المشكلة وإصلاح الخط.

وبالنسبة لسؤالك عن التعويض: أيوة الكلام ده مظبوط؛ وبما إن العطل عندك تجاوز 72 ساعة (بقالك 4 أيام)، فحضرتك مستحق لتعويض باقة نت للموبايل بحجم 5 جيجابايت وفقاً للسياسة المعتمدة.
--------------------------------------------------
